## Retrieval Augmented Generation (RAG)

In [2]:
%pip install chromadb sentence-transformers openai



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
documents = [
    {"id": "doc1", "text": "Acme Cloud offers three plans: Free, Pro, and Enterprise. The Free plan includes 5 GB of storage. The Pro plan costs 12 dollars per month and includes 500 GB of storage."},
    {"id": "doc2", "text": "Refund policy: Customers can request a full refund within 30 days of purchase. After 30 days, refunds are not available, but you can cancel anytime to avoid future charges."},
    {"id": "doc3", "text": "Acme Cloud support is available Monday to Friday, 9 AM to 6 PM IST. Enterprise customers get 24/7 phone support. Email support is at support@acmecloud.example."},
    {"id": "doc4", "text": "To reset your password, click 'Forgot password' on the login page. A reset link will be emailed to you and stays valid for 15 minutes."},
    {"id": "doc5", "text": "Acme Cloud data centers are located in Mumbai, Singapore, and Frankfurt. All data is encrypted at rest using AES-256 and in transit using TLS 1.3."},
    {"id": "doc6", "text": "The Enterprise plan includes single sign-on (SSO), audit logs, a 99.99 percent uptime SLA, and unlimited storage. Pricing is custom and requires contacting sales."},
]


In [2]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

vec = embedder.encode("Hello world")
print(len(vec))  # 384 numbers


/home/anura/Pari/workarea/python-virtual-environments/demo-backup-venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3578.79it/s]


384


In [ ]:
# dense vector
vec

array([-3.44772823e-02,  3.10231876e-02,  6.73496490e-03,  2.61089820e-02,
       -3.93620096e-02, -1.60302550e-01,  6.69240132e-02, -6.44145487e-03,
       -4.74505015e-02,  1.47588663e-02,  7.08753169e-02,  5.55274822e-02,
        1.91933401e-02, -2.62512919e-02, -1.01095345e-02, -2.69405376e-02,
        2.23074257e-02, -2.22266316e-02, -1.49692640e-01, -1.74931288e-02,
        7.67622842e-03,  5.43523021e-02,  3.25447577e-03,  3.17259766e-02,
       -8.46214220e-02, -2.94059832e-02,  5.15956581e-02,  4.81240824e-02,
       -3.31477099e-03, -5.82791679e-02,  4.19692844e-02,  2.22107526e-02,
        1.28188848e-01, -2.23389510e-02, -1.16562443e-02,  6.29284158e-02,
       -3.28763090e-02, -9.12260413e-02, -3.11753359e-02,  5.26995435e-02,
        4.70348410e-02, -8.42030346e-02, -3.00561916e-02, -2.07448304e-02,
        9.51776560e-03, -3.72172822e-03,  7.34330853e-03,  3.93243544e-02,
        9.32740197e-02, -3.78856319e-03, -5.27421013e-02, -5.80581911e-02,
       -6.86436752e-03,  

In [3]:
def chunk_text(text, chunk_size=200, overlap=40):
    """Split text into overlapping character chunks."""
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks


In [5]:
import time
for i, doc in enumerate(documents):
    print(i)
    print(len(doc["text"]))
    print(chunk_text(doc["text"], chunk_size=200, overlap=50))
    time.sleep(2)

0
169
['Acme Cloud offers three plans: Free, Pro, and Enterprise. The Free plan includes 5 GB of storage. The Pro plan costs 12 dollars per month and includes 500 GB of storage.', ' 500 GB of storage.']
1
172
['Refund policy: Customers can request a full refund within 30 days of purchase. After 30 days, refunds are not available, but you can cancel anytime to avoid future charges.', ' avoid future charges.']
2
159
['Acme Cloud support is available Monday to Friday, 9 AM to 6 PM IST. Enterprise customers get 24/7 phone support. Email support is at support@acmecloud.example.', '.example.']
3
134
["To reset your password, click 'Forgot password' on the login page. A reset link will be emailed to you and stays valid for 15 minutes."]
4
146
['Acme Cloud data centers are located in Mumbai, Singapore, and Frankfurt. All data is encrypted at rest using AES-256 and in transit using TLS 1.3.']
5
162
['The Enterprise plan includes single sign-on (SSO), audit logs, a 99.99 percent uptime SLA, and 

In [6]:
import chromadb

chroma = chromadb.Client()  # in-memory; use PersistentClient(path="./db") to save to disk
collection = chroma.get_or_create_collection(name="acme_docs")

In [11]:
ids, texts = [], []
for doc in documents:
    for i, chunk in enumerate(chunk_text(doc["text"], chunk_size=300, overlap=50)):
        ids.append(f"{doc['id']}_chunk{i}")
        texts.append(chunk)

embeddings = embedder.encode(texts).tolist()

collection.add(ids=ids, documents=texts, embeddings=embeddings)
print(f"Indexed {collection.count()} chunks")


Indexed 6 chunks


In [12]:
print(len(embeddings))

6


In [ ]:
#embeddings

[[-0.003962326794862747,
  -0.003785129636526108,
  0.00019036336743738502,
  0.04546240344643593,
  0.07811946421861649,
  0.0422925129532814,
  -0.0033836832735687494,
  0.07991328090429306,
  0.05343680456280708,
  0.1170143261551857,
  -0.013312260620296001,
  -0.0017332576680928469,
  0.0660833939909935,
  -0.04962712898850441,
  0.01882437989115715,
  -0.0019607844296842813,
  0.0034959211479872465,
  -0.1153445690870285,
  -0.010572760365903378,
  -0.05226540565490723,
  -0.029097529128193855,
  -0.046529460698366165,
  -0.012393700890243053,
  -0.04285652935504913,
  0.04095039144158363,
  -0.06665796786546707,
  0.008890854194760323,
  -0.021011075004935265,
  -0.0022238469682633877,
  -0.004820546135306358,
  0.06364519894123077,
  -0.018389767035841942,
  -0.004432633984833956,
  0.0825987309217453,
  0.046890076249837875,
  -0.05541409179568291,
  -0.06623627990484238,
  -0.05016795173287392,
  -0.03965776413679123,
  -0.06151435151696205,
  0.020096220076084137,
  0.009481

In [13]:
def retrieve(question, k=3):
    q_vec = embedder.encode(question).tolist()
    results = collection.query(query_embeddings=[q_vec], n_results=k)
    return results["documents"][0], results["distances"][0]

chunks, distances = retrieve("Can I get my money back after 40 days?")

print(chunks)

for c, d in zip(chunks, distances):
    print(f"[distance={d:.3f}] {c}\n")


['Refund policy: Customers can request a full refund within 30 days of purchase. After 30 days, refunds are not available, but you can cancel anytime to avoid future charges.', "To reset your password, click 'Forgot password' on the login page. A reset link will be emailed to you and stays valid for 15 minutes.", 'Acme Cloud support is available Monday to Friday, 9 AM to 6 PM IST. Enterprise customers get 24/7 phone support. Email support is at support@acmecloud.example.']
[distance=0.881] Refund policy: Customers can request a full refund within 30 days of purchase. After 30 days, refunds are not available, but you can cancel anytime to avoid future charges.

[distance=1.452] To reset your password, click 'Forgot password' on the login page. A reset link will be emailed to you and stays valid for 15 minutes.

[distance=1.614] Acme Cloud support is available Monday to Friday, 9 AM to 6 PM IST. Enterprise customers get 24/7 phone support. Email support is at support@acmecloud.example.



In [14]:
def build_prompt(question, context_chunks):
    context = "\n\n".join(f"- {c}" for c in context_chunks)
    return f"""Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know based on the provided documents."

Context:
{context}

Question: {question}

Answer:"""


In [15]:
import os
from openai import OpenAI

"""
llm = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

MODEL = "openai/gpt-4o-mini"
"""

llm = OpenAI(
    base_url="https://api.apinex.bond/v1",
    api_key=os.environ["OPENAI_API_KEY"],
)

MODEL = "free/deepseek-v4.1-flash"

def generate(prompt):
    response = llm.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content


In [16]:
question = "Can I get my money back after 40 days?"

prompt = build_prompt(question, chunks)

print(generate(prompt))

No, you cannot get a full refund after 40 days because refunds are only available within 30 days of purchase. However, you can cancel anytime after that to avoid future charges.


In [22]:
chunks

['Refund policy: Customers can request a full refund within 30 days of purchase. After 30 days, refunds are not available, but you can cancel anytime to avoid future charges.',
 "To reset your password, click 'Forgot password' on the login page. A reset link will be emailed to you and stays valid for 15 minutes.",
 'Acme Cloud support is available Monday to Friday, 9 AM to 6 PM IST. Enterprise customers get 24/7 phone support. Email support is at support@acmecloud.example.']